# Exercises XP – Week 3, Day 2: Data Preprocessing & Transformation
All exercises use the Titanic dataset (`train.csv`, in this same folder).
Order followed (as recommended): duplicates → missing values → feature engineering → outliers → encoding → scaling → age groups.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

df = pd.read_csv("train.csv")
df.head()

## 🌟 Exercise 1: Duplicate Detection and Removal

In [ ]:
rows_before = len(df)
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

rows_after = len(df)
print("Rows before:", rows_before, "| Rows after:", rows_after)

## 🌟 Exercise 2: Handling Missing Values
Different columns get different strategies, depending on what the data means:
- **Cabin** – about 77% missing, too much to fill reliably → fill with a constant `"Unknown"` (and keep the info "had a cabin or not").
- **Age** – about 20% missing, numeric → impute with the **median** (the median isn't pulled by extreme ages).
- **Embarked** – only 2 missing, categorical → impute with the **most frequent** value.

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Strategy 1 - constant value: Cabin
df["HasCabin"] = df["Cabin"].notnull().astype(int)
df["Cabin"] = df["Cabin"].fillna("Unknown")

# Strategy 2 - imputation with the median: Age (SimpleImputer from scikit-learn)
age_imputer = SimpleImputer(strategy="median")
df[["Age"]] = age_imputer.fit_transform(df[["Age"]])

# Strategy 3 - imputation with the most frequent value: Embarked
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Strategy 4 - removal (dropna), shown on a copy just to compare:
print("Rows if we simply dropped every row with a missing value:", len(pd.read_csv("train.csv").dropna()))

print("Missing values left:", df.isnull().sum().sum())

## 🌟 Exercise 3: Feature Engineering
- **FamilySize** = siblings/spouses (`SibSp`) + parents/children (`Parch`) + the passenger themself.
- **Title** extracted from the name (e.g. "Braund, **Mr.** Owen Harris"). Rare titles are grouped as `"Rare"`.
- **Title** is label-encoded here (no scaling yet).

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common_titles = ["Mr", "Mrs", "Miss", "Master"]
df.loc[~df["Title"].isin(common_titles), "Title"] = "Rare"

label_encoder = LabelEncoder()
df["Title_encoded"] = label_encoder.fit_transform(df["Title"])

print(df["Title"].value_counts())
df[["Name", "Title", "Title_encoded", "SibSp", "Parch", "FamilySize"]].head()

## 🌟 Exercise 4: Outlier Detection and Handling
First look at the distributions, then detect outliers with the **IQR** method and the **Z-score** method.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column="Fare", ax=axes[0])
df.boxplot(column="Age", ax=axes[1])
plt.suptitle("Before outlier treatment")
plt.show()

In [ ]:
# IQR method
for col in ["Fare", "Age"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: IQR outliers = {n_outliers} (outside {lower:.1f} to {upper:.1f})")

# Z-score method (|z| > 3)
for col in ["Fare", "Age"]:
    z = (df[col] - df[col].mean()) / df[col].std()
    print(f"{col}: Z-score outliers = {(z.abs() > 3).sum()}")

In [ ]:
# Look at the top quantiles to choose a capping threshold
print(df["Fare"].quantile([0.95, 0.98, 0.99, 1.0]))

Three ways of handling the outliers, compared:
1. **Quantile capping at 0.98** – values above the 98th percentile are set to that value (keeps every row).
2. **Log transformation** – `log(1 + Fare)` squeezes the long right tail (Fare is very skewed).
3. **Row removal** – drop rows outside the IQR bounds (loses data, shown for comparison only).

I keep **capping for Fare** in the main dataset, because it keeps all passengers and removes the extreme effect. Age outliers are realistic (old passengers), so I cap Age only lightly at the 0.99 quantile.

In [ ]:
df_before = df.copy()

# 1. Quantile capping
fare_cap = df["Fare"].quantile(0.98)
df["Fare"] = df["Fare"].clip(upper=fare_cap)
df["Age"] = df["Age"].clip(upper=df["Age"].quantile(0.99))

# 2. Log transformation (stored as a new column)
df["Fare_log"] = np.log1p(df_before["Fare"])

# 3. Row removal (only to compare)
q1, q3 = df_before["Fare"].quantile([0.25, 0.75])
iqr = q3 - q1
removed = df_before[(df_before["Fare"] >= q1 - 1.5 * iqr) & (df_before["Fare"] <= q3 + 1.5 * iqr)]
print("Rows after removal would be:", len(removed), "out of", len(df_before))

# Compare before and after
print(pd.DataFrame({"Fare before": df_before["Fare"].describe(),
                    "Fare after capping": df["Fare"].describe()}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(df_before["Fare"], bins=40)
axes[0].set_title("Fare – original")
axes[1].hist(df["Fare"], bins=40)
axes[1].set_title("Fare – capped at 0.98")
axes[2].hist(df["Fare_log"], bins=40)
axes[2].set_title("Fare – log transformed")
plt.show()

## 🌟 Exercise 6: Feature Encoding
*(Done before scaling, as the reminder in the instructions says.)*
- `Sex`, `Embarked` and `Title` are **nominal** (no natural order) → **one-hot encoding**.
- `Pclass` is **ordinal** (1st > 2nd > 3rd) and is already numeric, so it stays as it is.

In [ ]:
df = pd.get_dummies(df, columns=["Sex", "Embarked", "Title"], drop_first=False, dtype=int)
df.head()

## 🌟 Exercise 5: Data Standardization and Normalization
*(Done after outlier treatment and encoding.)*
- **Age** is roughly bell-shaped → **StandardScaler** (mean 0, std 1).
- **Fare** and **FamilySize** are skewed / bounded → **MinMaxScaler** (range 0 to 1).

In [ ]:
standard_scaler = StandardScaler()
df[["Age_scaled"]] = standard_scaler.fit_transform(df[["Age"]])

minmax_scaler = MinMaxScaler()
df[["Fare_scaled", "FamilySize_scaled"]] = minmax_scaler.fit_transform(df[["Fare", "FamilySize"]])

df[["Age_scaled", "Fare_scaled", "FamilySize_scaled"]].describe().round(2)

## 🌟 Exercise 7: Data Transformation for Age Feature

In [ ]:
bins = [0, 12, 18, 60, 100]
labels = ["Child", "Teen", "Adult", "Senior"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

print(df["AgeGroup"].value_counts())

age_dummies = pd.get_dummies(df["AgeGroup"], prefix="AgeGroup", dtype=int)
df = pd.concat([df, age_dummies], axis=1)

df[["Age", "AgeGroup"] + list(age_dummies.columns)].head()